In [155]:
import sys
import os
from pathlib import Path
import numpy as np
from SciExpeM_API.SciExpeM import SciExpeM
# path to extract_data.py and input data
sciexp_scripts_path = r"/Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts"
import importlib.util
extract_data_spec = importlib.util.spec_from_file_location('extract_data', os.path.join(sciexp_scripts_path, 'extract_data.py'))
extract_data = importlib.util.module_from_spec(extract_data_spec)
sys.modules['extract_data'] = extract_data
extract_data_spec.loader.exec_module(extract_data)
print('extract_data loaded from:', extract_data.__file__)
build_sciexp_objects_spec = importlib.util.spec_from_file_location('build_sciexp_objects', os.path.join(sciexp_scripts_path, 'build_sciexp_objects.py'))
build_sciexp_objects = importlib.util.module_from_spec(build_sciexp_objects_spec)
sys.modules['build_sciexp_objects'] = build_sciexp_objects
build_sciexp_objects_spec.loader.exec_module(build_sciexp_objects)
print('build_sciexp_objects loaded from:', build_sciexp_objects.__file__)

my_sciexpem = SciExpeM(username='YOUR_USERNAME', password='YOUR_PASSWORD', secure=False, port=8080
                       )
# database: porta 5432 all'indirizzo 127.0.0.1
from SciExpeM_API.Models import *
CONVERT_TO_BAR = {'atm': 1.01325, 'bar': 1., 'Torr': 0.00133322, 'Pa': 1e-5}
sp_table = {}

extract_data loaded from: /Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts/extract_data.py
build_sciexp_objects loaded from: /Users/lpratalimaffei/Library/CloudStorage/OneDrive-PolitecnicodiMilano/Luna/Universita/RICERCA/SCIEXPEM/SciExpeM_API/Examples/scripts/build_sciexp_objects.py
Attention. SciExpeM is a singleton.


### FilePaper
 - description*
 - reference_doi*
 - author*
 - title*
 - year*
 - volume
 - page
 - journal

In [156]:
#TEMPLATE
file_paper = FilePaper(reference_doi="10.1016/J.PROCI.2020.07.130",
                       author="Kukkadapu, G.;W agnon, S. W.; Pitz, W. J.; Hansen, N.",
                       title="Identification of the molecular-weight growth reaction network in counterflow flames of the C3H4 isomers allene and propyne",
                       year=2021,
                       description="G. Kukkadapu, S.W. Wagnon, W.J. Pitz, N. Hansen, Identification of the molecular-weight growth reaction network in counterflow flames of the C3H4 isomers allene and propyne, Proc. Combust. Inst. 38 (2021) 1477–1485.",
                       )


### OpenSMOKE input file if available

In [157]:
datapath = os.path.join(sciexp_scripts_path,'data')
osinputname = 'CF_inputOS.dic'
# WARNING LIST OF SPECIES MUST HAVE THE NAMES OF THE MECH YOU ARE GOING TO SIMULATE - OTHERWISE, NEED REPLACEMENT
inputstr, extrainfo = extract_data.process_osinput(datapath, osinputname, profiles=False, flameinfo=True)  
# LA LISTA DI OUTPUTSPECIES DEVE AVERE SPECIE NEL MECCANISMO CHE SIMULERAI

In [158]:
print(extrainfo)

{'profileinfo': {}, 'commonprop': {'fuel velocity': ['22.12', 'cm/s'], 'oxidizer velocity': ['22.12', 'cm/s'], 'fuel temperature': ['300.1', 'K'], 'oxidizer temperature': ['300.1', 'K'], 'peak mixture temperature': ['2000', 'K'], 'pressure': ['0.933', 'atm'], 'length': ['1.2', 'cm']}, 'character': {}, 'molefractions': [['C3H4-A', 'AR'], ['0.069', '0.931']], 'oxidizermolefractions': [['O2', 'AR'], ['0.2', '0.8']]}


### Common properties

- name
- units
- value
- source_type

In [159]:
# ONLY COUNTERFLOW: stoich mix fraction
# Z=1/(1+m*Xf/Xo) , m=ox/fuel stoich, in mass; Xf: fuel mass frac in fuel inlet; Xo=ox. mass frac in ox inlet
stoich_mix_fraction = extract_data.stoichiometric_mixture_fraction(extrainfo['molefractions'], extrainfo['oxidizermolefractions'])
print(stoich_mix_fraction)

0.43014193770681475


In [160]:
########### ONLY COUNTERFLOW

fuel_v, ufv = extrainfo['commonprop']['fuel velocity']
oxid_v, uov = extrainfo['commonprop']['oxidizer velocity']
length, ul = extrainfo['commonprop']['length']
assert ufv.split('/')[0] == uov.split('/')[0] == ul #check all in cm
strain_rate = (float(fuel_v) + float(oxid_v)) / float(length) #(|Vox|+|Vfuel|)/length
# others
sourcetype = 'reported'
commonprop = []
for name, values in extrainfo['commonprop'].items():
    if name == 'velocity': # velocity is estimated from cold gas velocity and inlet T
        ci = CommonProperty(name=name, units=values[1], value=values[0], source_type='calculated')
    elif name == 'peak mixture temperature':
        continue
    ci = CommonProperty(name=name, units=values[1], value=values[0], source_type=sourcetype)
    commonprop.append(ci)
    
# ADD OTHER COMMON PROPERTIES
#ci = CommonProperty(name='temperature', units='K', value='1120', source_type='reported')
#commonprop.append(ci)
############# DO NOT EDIT
# extract pressure
if 'pressure' in extrainfo['commonprop'].keys():
    Pval, Punit = extrainfo['commonprop']['pressure']
    P = float(Pval) * CONVERT_TO_BAR[Punit]

cr = CommonProperty(name = 'stoichiometric mixture fraction', units = 'unitless', value = stoich_mix_fraction, source_type= 'calculated'  )
commonprop.append(cr)
cr = CommonProperty(name = 'strain rate', units = 's-1', value = strain_rate, source_type= 'calculated'  )
commonprop.append(cr)

In [161]:
for c in commonprop:
    print(c.serialize())

{'name': 'fuel velocity', 'units': 'cm/s', 'value': '22.12', 'source_type': 'reported'}
{'name': 'oxidizer velocity', 'units': 'cm/s', 'value': '22.12', 'source_type': 'reported'}
{'name': 'fuel temperature', 'units': 'K', 'value': '300.1', 'source_type': 'reported'}
{'name': 'oxidizer temperature', 'units': 'K', 'value': '300.1', 'source_type': 'reported'}
{'name': 'pressure', 'units': 'atm', 'value': '0.933', 'source_type': 'reported'}
{'name': 'length', 'units': 'cm', 'value': '1.2', 'source_type': 'reported'}
{'name': 'stoichiometric mixture fraction', 'units': 'unitless', 'value': np.float64(0.43014193770681475), 'source_type': 'calculated'}
{'name': 'strain rate', 'units': 's-1', 'value': 36.86666666666667, 'source_type': 'calculated'}


### Initial Species

- name
- units
- value
- source_type
- configuration

In [162]:
################ COUNTERFLOW
# THIS REFERS TO THE SIMULATION
# PREMIXED IS DEFAULT AND MUST BE INDICATED UNLESS IT'S A CF FLAME
comp_unit = 'mole fraction'
srctype = 'reported'
################# do not edit
inspecies, fuelobjs = build_sciexp_objects.build_initial_species(
    my_sciexpem=my_sciexpem,
    molefractions=extrainfo['molefractions'],
    oxidizer_molefractions=extrainfo['oxidizermolefractions'],
    source_type=srctype,
    units=comp_unit,
)


Species: ['C3H4-A', 'AR', 'O2', 'AR']
Fuels: ['C3H4-A']
Mole Frac: ['0.069', '0.931', '0.2', '0.8']
config: ['fuel', 'fuel', 'oxidizer', 'oxidizer']


### Data columns

- name
- label (not comuplsory)
- species_object
- units
- data
- dg_id 
- dg_label
- source_type


In [163]:
## if species indicated as stoichiometry: sum of all isomers available on sciexpem
sp_table = {
    'C14H8': 'C12H7C2H-1+C12H7C2H-5+C10H6(C2H)2_298+C10H6(C2H)2',
    'C9H8':  'INDENE+C6H5C3H3+C6H5C3H3_502+C6H5CCCH3',
    'C13H10': 'FLUORENE+BENZINDENE+BENZINDENE-1HF+BENZINDENE-3HE+PHENALENE',
    'C10H7CH3': 'C10H7CH3+C10H7CH3_106',
}
# if not needed : empty dictionary

In [164]:
# read data
datafile = os.path.join(sciexp_scripts_path, 'data', 'CF_data.txt')
df_data = extract_data.readdata(datafile, delzero=True)
# process data
srctype = 'digitized'
label = 'experimental_data'
#x = ['temperature', 'K']
#x = ['time', 's'] # NB must be 'residence time' for concentration time profile, but only works with 'time' lol
#y = ['concentration', 'mol/cm3']
x = ['distance', 'cm']
y = ['composition', 'mole fraction']
list_exclude_species = [] #optional
uncert_x = []
#uncert_x = [30, 'absolute']
#uncert_y = [0.2, 'relative']
uncert_y = [0.3, 'relative'] # default uncertainty if not available in datafile

#################### DO NOT EDIT################
TINF, TSUP = extract_data.temperature_bounds(
    extrainfo,
    x=x,
    df_data=df_data,
)
print('T max and min: {} - {} K'.format(TINF, TSUP))

datacols = build_sciexp_objects.build_data_columns(
    df_data=df_data,
    my_sciexpem=my_sciexpem,
    x=x,
    y=y,
    source_type=srctype,
    label=label,
    sp_table=sp_table,
    list_exclude_species=list_exclude_species,
    uncert_x=uncert_x,
    uncert_y=uncert_y,
)

T max and min: 300.1 - 2000.0 K
C16H10 [<Species (31)>]
C3H4-A [<Species (15)>]
CH4 [<Species (6)>]
C2H2 [<Species (51)>]
C6H6 [<Species (39)>]
species C12H10 not as preferred name: alternative names searched
C12H10 [<Species (17)>]
C14H10 [<Species (29)>]
C13H10 [<Species (34)>, <Species (151)>, <Species (573)>, <Species (574)>, <Species (149)>]
C15H10 [<Species (154)>]
C7H8 [<Species (50)>]
C6H5C2H [<Species (45)>]
C10H8 [<Species (27)>]
C12H8 [<Species (28)>]
C10H7CH3 [<Species (22)>, <Species (106)>]
species C10H6(C2H)2_298 not as preferred name: alternative names searched
C14H8 [<Species (476)>, <Species (508)>, <Species (297)>]
C9H8 [<Species (35)>, <Species (486)>, <Species (502)>, <Species (478)>]


### Assemble the experiment

In [165]:
e = Experiment(reactor='flame', # [flow reactor, flame, stirred reactor ]
               experiment_type='burner stabilized flame speciation measurement',
               file_paper=file_paper,
               reactor_modes=['counterflow'], # da mettere in tutte le fiamme # coflow, couterflow, premixed, burner stabilized stagnation
               #reactor_modes=['premixed'], # da mettere in tutte le fiamme # coflow, couterflow, premixed, burner stabilized stagnation
               data_columns=datacols, 
               initial_species=inspecies, 
               common_properties=commonprop,
               os_input_file=inputstr,
               t_inf = TINF, t_sup = TSUP,
               p_inf = P, p_sup = P,
               phi_inf = 0, phi_sup = 100.,
               # fuels=fuels,
               fuels_object = fuelobjs, # devi passargli gli id delle specie che sono fuels
               comment = 'uncertainty: small species: 30%; larger aromatics: 2; no PICS for large aromatics: 4; data NOT ISOMER-RESOLVED: CONTRIBUTIONS OF ALIPHATIC SPECIES FOR C9H8 AND C13H10 IS CERTAIN; THE OTHERS ARE SUMS'
               #comment = 'technically not isothermal but simulated as such - T almost unvaried'
               #comment = 'C4H2 plot unclear, order might be 1e6 or 1e8 (person who digitized assumed 1e8); bath gas varies with experimental run (Ne, He), but should not affect the results'
               )

In [76]:
print(fuelobjs)

[<Species (15)>]


In [ ]:
e.serialize()
# check that everything is ok

### Send Experiment

In [166]:
my_sciexpem.insertElement(e, verbose=True)

400 Client Error: Bad Request for url: http://sciexpem.chem.polimi.it:8080/ExperimentManager/API/insertElement


HTTP ERROR 400 -> insertElement: Experiment Consistency Error. Uncertainty relative data must be 0 <= x <= 1.


In [11]:
e = my_sciexpem.filterDatabase(model_name='Experiment', id = 3828)[0]

In [30]:
# aggiungi cella per verificare da sciexpem_API
e = my_sciexpem.filterDatabase(model_name='Experiment', id = 2831)[0]
print(e)
for c in e.common_properties:
    print(c.name)

<Experiment (2831)>
pressure
velocity
residence time
length
